# Scenario comparison

Compares month-end **nominal** net worth across any number of scenarios against a
baseline, and finds each variant's break-even month — the point after which it
stays ahead for good.

Pick a baseline and up to 4 variant scenarios below — every chart and table in
this notebook updates automatically as the selection changes, no re-running
required. Each must already have a run (`analysis sim load <name> && analysis
sim run <name>`); the newest run for each is used.

> **Note:** after running cells, `git status` will show this notebook as modified
> even with no real changes — see [notebooks/README.md](./README.md#git-status-shows-a-notebook-as-modified-after-running-it).



In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

from IPython.display import display

from analysis.sim.loader import list_scenario_names
from analysis.sim.notebook_ui.pickers import multi_scenario_picker

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = Path(os.environ.get("ANALYSIS_DATA_DIR", REPO / "data" / "shared" / "sim"))
MAX_VARIANTS = 4

_available = list_scenario_names(DATA_DIR)
_default_baseline = "current" if "current" in _available else _available[0]
_default_variants = [name for name in _available if name != _default_baseline][:MAX_VARIANTS]

# PICKER holds the baseline/variant widgets; CONNECTIONS holds the live
# RunConnections, refreshed in place whenever either picker changes.
PICKER, CONNECTIONS = multi_scenario_picker(
    DATA_DIR, _default_baseline, _default_variants, MAX_VARIANTS
)
baseline_picker, variant_picker = PICKER.children[0].children
display(PICKER)


## Are these runs comparable?

A comparison is only meaningful if both runs came from the same code and the same
data version — otherwise a difference here could be an unrelated edit rather than
the scenario change under test.


In [ ]:
from analysis.sim.notebook_ui.compare_plots import comparability_widget

display(comparability_widget(CONNECTIONS, baseline_picker, variant_picker))


## What actually differs?

Each scenario inherits everything from its parent except the rows it overrides, so
this is the whole mechanism behind any difference in the charts below. If a rule
you expected to change isn't listed, the override didn't take effect.

Two sources of difference are shown separately:

- **Rules** — as they stand at the *end* of the run. A calendar event that edits a
  rule mid-run (`set_amount`) shows up here as a changed amount, because the run
  snapshot records the post-event state.
- **Calendar events** — one-time interventions, including the benefit cut.


In [ ]:
from analysis.sim.notebook_ui.compare_plots import rules_and_events_widget

display(rules_and_events_widget(CONNECTIONS, baseline_picker, variant_picker))


## Net worth over time

Nominal dollars. (`net_worth_constant_cpi` is available in each run if you want
the inflation-adjusted view later.)


In [ ]:
from analysis.sim.notebook_ui.compare_plots import net_worth_compare_widget

display(net_worth_compare_widget(CONNECTIONS, baseline_picker, variant_picker))


## Break-even

**Break-even** is the first month after which the variant stays ahead for the rest
of the scenario — not merely the first month it pokes above zero, which can happen
transiently when a lumpy payment lands in one scenario before the other.


In [ ]:
from analysis.sim.notebook_ui.compare_plots import breakeven_widget

display(breakeven_widget(CONNECTIONS, baseline_picker, variant_picker))


## Horizon sensitivity — how much does this matter, and by when?

A claim-timing decision only pays off if you live long enough to collect on it, so
the honest question isn't "what's the difference at 2060" but "what's the
difference **if we stop looking at year X**". Every point on these curves answers
that: the gap at year X *is* the impact of the decision if the money stops
mattering then.

The two panels are deliberately on different scales, because they answer different
questions:

- **The decision** — what waiting is worth. Small, slow, and it starts negative.
- **The risk** — what a Trust Fund shortfall costs. Large, and unrelated to the
  claim date.

If the decision is worth little at your realistic horizons, then taking the
benefit earlier for certainty and convenience costs you very little.


In [ ]:
from analysis.sim.notebook_ui.compare_plots import horizon_widget

# The pairs worth contrasting: (scenario, its counterfactual). Edit for other questions.
COMPARISONS = {
    "the decision": {
        "wait — no cut": ("january", "current"),
        "wait — with cut": ("january-cut", "current-cut"),
    },
    "the risk": {
        "cut — claiming now": ("current-cut", "current"),
        "cut — waiting": ("january-cut", "january"),
    },
}

display(horizon_widget(CONNECTIONS, baseline_picker, variant_picker, COMPARISONS))


In [ ]:
import ipywidgets as widgets
import pandas as pd
from IPython.display import Markdown, clear_output, display

from analysis.sim.connection import year_end_values
from analysis.sim.notebook_ui.compare_plots import (
    comparison_frame, filter_comparisons, has_any_comparisons, net_worth_by_month,
)

_horizon_detail_out = widgets.Output()


def _render_horizon_detail(*_):
    with _horizon_detail_out:
        clear_output(wait=True)
        if not CONNECTIONS.ok:
            return

        comparisons = filter_comparisons(COMPARISONS, CONNECTIONS.scenarios)
        if not has_any_comparisons(comparisons):
            display(Markdown(
                "_None of the configured `COMPARISONS` pairs apply to this selection "
                "— see the horizon-sensitivity chart above._"
            ))
            return

        nw = net_worth_by_month(CONNECTIONS)
        compare = comparison_frame(nw, comparisons)

        # Year-end value of each comparison, and the pot it is a share of.
        horizon = year_end_values(compare)
        horizon.index.name = "if we look out to"
        pot = year_end_values(nw[CONNECTIONS.baseline])

        def _horizon_table(years, frame):
            table = frame.loc[years].copy()
            for label in compare.columns:
                table[f"{label} (% of pot)"] = 100 * frame.loc[years, label] / pot.loc[years]
            return table.style.format(
                {c: ("{:+.2f}%" if c.endswith("(% of pot)") else "${:+,.0f}")
                 for c in table.columns}
            )

        outstanding = pd.DataFrame({
            name: CONNECTIONS.connections[name].outstanding_tax_by_year()
            for name in CONNECTIONS.scenarios
        }).fillna(0)
        nw_year_end = year_end_values(nw)
        settled = nw_year_end.sub(outstanding.reindex(nw_year_end.index).fillna(0), axis=0)
        compare_settled = comparison_frame(settled, comparisons)
        compare_settled.index.name = "if we look out to"

        shift = (compare_settled - horizon).abs().max(axis=1)
        moved = shift[shift >= 1].index
        display(Markdown(
            f"Settling changes the answer in **{len(moved)} of {len(horizon)}** years"
            + (f" ({', '.join(str(y) for y in moved)}):" if len(moved) else ".")
        ))
        if len(moved):
            display(
                pd.concat(
                    {"as-of": horizon.loc[moved], "settled": compare_settled.loc[moved]},
                    axis=1,
                ).style.format("${:+,.0f}")
            )

        marks = list(range(horizon.index.min() + 4, horizon.index.max() + 1, 5))
        display(Markdown("**At five-year horizons** (tax owed settled):"))
        display(_horizon_table(marks, compare_settled))

        def _at_year(year, settle_tax=True):
            """Net worth already reflects tax paid; `settle_tax` also clears what is owed."""
            frame = compare_settled if settle_tax else horizon
            basis = "after settling the year's tax bill" if settle_tax else "as of Dec 31"
            lines = [
                f"**If the money stops mattering at the end of {year}** "
                f"({basis}; net worth ${pot.loc[year]:,.0f}):",
                "",
            ]
            lines += [
                f"- {label}: **${value:+,.0f}** ({100 * value / pot.loc[year]:+.2f}% of the pot)"
                for label, value in frame.loc[year].items()
            ]
            display(Markdown("\n".join(lines)))

        year_slider = widgets.IntSlider(
            min=int(horizon.index.min()), max=int(horizon.index.max()),
            value=min(2036, int(horizon.index.max())), description="horizon",
            continuous_update=False,
        )
        settle_checkbox = widgets.Checkbox(value=True, description="settle tax owed")
        display(widgets.VBox([
            widgets.HBox([year_slider, settle_checkbox]),
            widgets.interactive_output(
                _at_year, {"year": year_slider, "settle_tax": settle_checkbox}
            ),
        ]))


baseline_picker.observe(_render_horizon_detail, names="value")
variant_picker.observe(_render_horizon_detail, names="value")
_render_horizon_detail()
display(_horizon_detail_out)


### Settling the tax bill at the horizon

Net worth above is after-tax — the engine pays tax as a real transfer — but a tax
year settles the *following* April. So at the end of year X each scenario is
carrying an unpaid liability (or an unclaimed refund) for year X, and net worth
flatters whichever scenario owes more.

Stopping at year X in real life means that bill still has to be paid, so the
comparison below deducts each scenario's outstanding balance. It only moves the
answer where the two scenarios owe *different* amounts — which is exactly the 2026
wage year that motivated the question.


## Year-end detail


In [ ]:
from analysis.sim.notebook_ui.compare_plots import year_end_widget

display(year_end_widget(CONNECTIONS, baseline_picker, variant_picker))


## What drives it: Social Security received

Gross benefit (including the Medicare premiums withheld before deposit), so the
two scenarios are compared on the same basis. The cumulative view is the classic
claim-timing picture: the later claim starts lower and crosses over.


In [ ]:
from analysis.sim.notebook_ui.compare_plots import ss_gross_widget

display(ss_gross_widget(CONNECTIONS, baseline_picker, variant_picker))


## Tax impact

Claiming earlier stacks Social Security on top of whatever else is taxable that
year. In a year with wages, that is expensive twice over: the benefit itself is
taxable *and* the wages push provisional income past the thresholds, so a larger
fraction of the benefit gets taxed (up to 85%).

Liability is attributed to the **tax year it was earned in**; the cash settles the
following April, so a difference shown for year *Y* leaves the bank account in
*Y+1*. Net worth already reflects this — the table below is the explanation, not
an additional effect.


In [ ]:
from analysis.sim.notebook_ui.compare_plots import tax_impact_widget

display(tax_impact_widget(CONNECTIONS, baseline_picker, variant_picker))
